In [ ]:
# =======================================================================
# CÉLULA 1 — Instalação de dependências
# =======================================================================
!pip uninstall torchvision -y
!pip install -q -U transformers datasets peft trl accelerate
!pip install -q rouge-score nltk evaluate
!pip install -q google-generativeai
!pip install -q "torchao>=0.16.0"

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.9 MB/s eta 0:00:00


In [ ]:
# =======================================================================
# CÉLULA 2 — Imports
# =======================================================================
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import evaluate
import nltk
import google.generativeai as genai
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, StoppingCriteria, StoppingCriteriaList
from peft import PeftModel
from tqdm import tqdm
import json

nltk.download("wordnet")
nltk.download("punkt")
nltk.download("punkt_tab")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# =======================================================================
# CÉLULA 3 — Configurações gerais
# =======================================================================
GEMINI_API_KEY = "sua_chave_aqui"
genai.configure(api_key=GEMINI_API_KEY)

MODEL_ID = "microsoft/bitnet-b1.58-2B-4T-bf16"
LORA_PATH = "./modelo_final_mediasum_v2"
OUTPUT_DIR = "./avaliacao_mediasum_v2_ft"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_AMOSTRAS_AUTO = 300       # Mude para 300 quando for rodar completo
N_AMOSTRAS_MANUAL = 40      # Mude para 40 quando for rodar completo

SEED = 42

In [ ]:
# =======================================================================
# CÉLULA 4 — Upload dos pesos LoRA e backup do baseline
# =======================================================================
from google.colab import files
import zipfile

print("Faça upload do backup_mediasum_v2_completo.zip (pesos LoRA)")
uploaded = files.upload()
with zipfile.ZipFile("backup_mediasum_v2_completo.zip", "r") as zip_ref:
    zip_ref.extractall("./")
print(f"Pesos LoRA disponíveis em: {LORA_PATH}")

print("\nFaça upload do backup_baseline_mediasum_v2.zip (índices do baseline)")
uploaded = files.upload()
with zipfile.ZipFile("backup_baseline_mediasum_v2.zip", "r") as zip_ref:
    zip_ref.extractall("./")
print("Índices do baseline disponíveis!")

Faça upload do backup_mediasum_v2_completo.zip (pesos LoRA)


Saving backup_mediasum_v2_completo.zip to backup_mediasum_v2_completo.zip
Pesos LoRA disponíveis em: ./modelo_final_mediasum_v2

Faça upload do backup_baseline_mediasum_v2.zip (índices do baseline)


Saving backup_baseline_mediasum_v2.zip to backup_baseline_mediasum_v2.zip
Índices do baseline disponíveis!


In [ ]:
# =======================================================================
# CÉLULA 5 — Download e carregamento do dataset
# =======================================================================
from huggingface_hub import hf_hub_download
import zipfile as zf

print("Baixando test_data.zip do MediaSum...")
zip_path = hf_hub_download(
    repo_id="ccdv/mediasum",
    filename="test_data.zip",
    repo_type="dataset"
)
extract_path = "./mediasum_data/test"
os.makedirs(extract_path, exist_ok=True)
with zf.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)
print("-> Extraído com sucesso!")

dados_test = []
with open("./mediasum_data/test/test_data.txt", "r") as f:
    for linha in f:
        item = json.loads(linha)
        texto = "\n".join([
            f"{speaker}: {fala}"
            for speaker, fala in zip(item["speaker"], item["utt"])
        ])
        resumo = item["summary"]

        if not texto or not resumo:
            continue

        if len(texto.split()) <= 1700:
            dados_test.append({"document": texto, "summary": resumo})

test_set_completo = Dataset.from_list(dados_test)

if N_AMOSTRAS_AUTO is not None:
    test_set = test_set_completo.shuffle(seed=SEED).select(range(N_AMOSTRAS_AUTO))
else:
    test_set = test_set_completo

print(f"-> {len(test_set_completo)} instâncias no test set completo")
print(f"-> {len(test_set)} instâncias carregadas para avaliação")

Baixando test_data.zip do MediaSum...
-> Extraído com sucesso!
-> 6528 instâncias no test set completo
-> 300 instâncias carregadas para avaliação


In [ ]:
# =======================================================================
# CÉLULA 6 — Função de inferência
# =======================================================================
class StopOnToken(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids

    def __call__(self, input_ids, scores, **kwargs):
        for stop_id in self.stop_token_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False

template = (
    "### Instruction:\nWrite a short news headline summarizing the main topic of this interview in 16 words or less.\n\n"
    "### Interview Transcript:\n{document}\n\n"
    "### Headline:\n"
)

def gerar_resumos(model, tokenizer, dataset, modo="finetuned"):
    resumos_gerados = []
    resumos_referencia = []
    dialogos = []

    stop_sequences = ["#####", "####", "###", "### Instruction", "### Interview",
                      "def ", "```", "Note:", "---"]
    stop_ids = []
    for seq in stop_sequences:
        ids = tokenizer.encode(seq, add_special_tokens=False)
        if ids:
            stop_ids.append(ids[0])
    stopping_criteria = StoppingCriteriaList([StopOnToken(stop_ids)])

    for item in tqdm(dataset, desc=f"Gerando resumos ({modo})"):
        prompt = template.format(document=item["document"])

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
                stopping_criteria=stopping_criteria
            )

        gerado = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        # Limpeza pós-geração
        for seq in stop_sequences:
            if seq in gerado:
                gerado = gerado[:gerado.index(seq)].strip()
        gerado = gerado.replace("**", "").strip()

        resumos_gerados.append(gerado)
        resumos_referencia.append(item["summary"])
        dialogos.append(item["document"])

    return dialogos, resumos_referencia, resumos_gerados

In [ ]:
# =======================================================================
# CÉLULA 7 — Funções de métricas automáticas
# =======================================================================
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")

def calcular_bertscore_manual(referencias, gerados):
    referencias = [str(r) if r is not None else "" for r in referencias]
    gerados = [str(g) if g is not None else "" for g in gerados]

    model_name = "roberta-large"
    tokenizer_bert = AutoTokenizer.from_pretrained(model_name)
    model_bert = AutoModel.from_pretrained(model_name).to("cuda")
    model_bert.eval()

    scores_p, scores_r, scores_f1 = [], [], []

    for ref, gen in tqdm(zip(referencias, gerados), total=len(referencias), desc="BERTScore"):
        if not ref.strip() or not gen.strip():
            scores_p.append(0.0)
            scores_r.append(0.0)
            scores_f1.append(0.0)
            continue

        with torch.no_grad():
            ref_tokens = tokenizer_bert(ref, return_tensors="pt", truncation=True, max_length=512).to("cuda")
            gen_tokens = tokenizer_bert(gen, return_tensors="pt", truncation=True, max_length=512).to("cuda")

            ref_emb = model_bert(**ref_tokens).last_hidden_state.squeeze(0)
            gen_emb = model_bert(**gen_tokens).last_hidden_state.squeeze(0)

            ref_emb = ref_emb / ref_emb.norm(dim=-1, keepdim=True)
            gen_emb = gen_emb / gen_emb.norm(dim=-1, keepdim=True)

            sim = torch.matmul(gen_emb, ref_emb.T)

            P = sim.max(dim=1).values.mean().item()
            R = sim.max(dim=0).values.mean().item()
            F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0

            scores_p.append(P)
            scores_r.append(R)
            scores_f1.append(F1)

    del model_bert
    torch.cuda.empty_cache()

    return torch.tensor(scores_p), torch.tensor(scores_r), torch.tensor(scores_f1)

def calcular_metricas(referencias, gerados, prefixo=""):
    print(f"\nCalculando métricas {prefixo}...")

    gerados_clean = [g if g.strip() else "empty" for g in gerados]

    rouge_result = rouge.compute(
        predictions=gerados_clean,
        references=referencias,
        use_stemmer=True
    )
    bleu_result = bleu.compute(
        predictions=gerados_clean,
        references=[[r] for r in referencias]
    )
    meteor_result = meteor.compute(
        predictions=gerados_clean,
        references=referencias
    )

    P, R, F1 = calcular_bertscore_manual(referencias, gerados)

    resultados = {
        "ROUGE-1": round(rouge_result["rouge1"], 4),
        "ROUGE-2": round(rouge_result["rouge2"], 4),
        "ROUGE-L": round(rouge_result["rougeL"], 4),
        "BLEU": round(bleu_result["bleu"], 4),
        "METEOR": round(meteor_result["meteor"], 4),
        "BERTScore-P": round(P.mean().item(), 4),
        "BERTScore-R": round(R.mean().item(), 4),
        "BERTScore-F1": round(F1.mean().item(), 4),
    }

    return resultados

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# =======================================================================
# CÉLULA 8 — Seleção estratificada das amostras
# =======================================================================
def selecionar_amostras_estratificadas(df, n=40, seed=SEED):
    df = df.copy()
    df_sorted = df.sort_values("rouge_l_individual").reset_index(drop=True)
    n_total = len(df_sorted)
    n_por_faixa = n // 3

    baixo = df_sorted.iloc[:n_total//3].sample(n=n_por_faixa, random_state=seed)
    medio = df_sorted.iloc[n_total//3:2*n_total//3].sample(n=n_por_faixa, random_state=seed)
    alto = df_sorted.iloc[2*n_total//3:].sample(n=n - 2*n_por_faixa, random_state=seed)

    amostras = pd.concat([baixo, medio, alto]).sample(frac=1, random_state=seed)
    return amostras

In [ ]:
# =======================================================================
# CÉLULA 9 — G-Eval com Gemini (desativado por ora)
# =======================================================================
import time

def geval_gemini(dialogo, resumo_referencia, resumo_gerado):
    prompt = f"""You are an expert evaluator of text summarization quality.
You will evaluate a generated summary based on a transcript and a reference summary.
Score each dimension from 1 to 5, where 1 is very poor and 5 is excellent.

TRANSCRIPT:
{dialogo}

REFERENCE SUMMARY:
{resumo_referencia}

GENERATED SUMMARY:
{resumo_gerado}

Evaluate the generated summary on these 4 dimensions:
- Fluency (1-5): Is the text grammatically correct and natural to read?
- Coherence (1-5): Are ideas logically organized and well connected?
- Consistency (1-5): Does the summary avoid contradicting or hallucinating information from the transcript?
- Relevance (1-5): Does the summary capture the most important information?

Respond ONLY with a JSON object in this exact format, no extra text:
{{"fluency": <score>, "coherence": <score>, "consistency": <score>, "relevance": <score>}}"""

    for tentativa in range(5):
        try:
            gemini_model = genai.GenerativeModel("gemini-2.0-flash-lite")
            response = gemini_model.generate_content(prompt)
            texto = response.text.strip()
            texto = texto.replace("```json", "").replace("```", "").strip()
            scores = eval(texto)
            return scores
        except Exception as e:
            if "429" in str(e):
                espera = 30 * (tentativa + 1)
                print(f"Quota atingida (tentativa {tentativa+1}/5), aguardando {espera}s...")
                time.sleep(espera)
            else:
                print(f"Erro no G-Eval: {e}")
                return {"fluency": None, "coherence": None, "consistency": None, "relevance": None}

    print("G-Eval falhou após 5 tentativas.")
    return {"fluency": None, "coherence": None, "consistency": None, "relevance": None}


def calcular_geval(amostras_df):
    print("\nRodando G-Eval com Gemini nas amostras selecionadas...")
    scores = []
    for _, row in tqdm(amostras_df.iterrows(), total=len(amostras_df)):
        score = geval_gemini(
            row["dialogo"],
            row["resumo_referencia"],
            row["resumo_gerado"]
        )
        scores.append(score)

    amostras_df["geval_fluency"] = [s["fluency"] for s in scores]
    amostras_df["geval_coherence"] = [s["coherence"] for s in scores]
    amostras_df["geval_consistency"] = [s["consistency"] for s in scores]
    amostras_df["geval_relevance"] = [s["relevance"] for s in scores]

    return amostras_df

In [ ]:
# =======================================================================
# CÉLULA 10 — Inferência fine-tuning
# =======================================================================
print("=========================================================")
print("INFERÊNCIA COM FINE-TUNING (MediaSum v2 LoRA)")
print("=========================================================")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model_ft = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa"
)
model_ft = PeftModel.from_pretrained(model_ft, LORA_PATH)
model_ft.eval()

dialogos, refs, gerados_ft = gerar_resumos(
    model_ft, tokenizer, test_set, modo="finetuned"
)

df_ft = pd.DataFrame({
    "dialogo": dialogos,
    "resumo_referencia": refs,
    "resumo_gerado": gerados_ft
})
df_ft.to_csv(f"{OUTPUT_DIR}/resultados_finetuning.csv", index=False)
print(f"Inferência salva: {len(df_ft)} resumos")

del model_ft
torch.cuda.empty_cache()

INFERÊNCIA COM FINE-TUNING (MediaSum v2 LoRA)


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

Gerando resumos (finetuned): 100%|██████████| 300/300 [33:57<00:00,  6.79s/it]

Inferência salva: 300 resumos


In [ ]:
# =======================================================================
# CÉLULA 10A — Verificação de alucinações (5 exemplos)
# =======================================================================
df = pd.read_csv(f"{OUTPUT_DIR}/resultados_finetuning.csv")
for i, row in df.head(5).iterrows():
    print(f"\n--- Exemplo {i+1} ---")
    print(f"TRANSCRIÇÃO (primeiros 300 chars):\n{row['dialogo'][:300]}...")
    print(f"\nREFERÊNCIA:\n{row['resumo_referencia']}")
    print(f"\nGERADO:\n{row['resumo_gerado']}")
    print("="*60)


--- Exemplo 1 ---
TRANSCRIÇÃO (primeiros 300 chars):
M. O'BRIEN: You know, when I played "Monopoly," I never liked having the utilities. Did you like having the utilities?
ANDY SERWER, "FORTUNE": Steady dividend. Steady.
M. O'BRIEN: It would apparently be a good buy right now, right, Andy Serwer?
SERWER: Absolutely. I'll tell you what's going on right...

REFERÊNCIA:
Utility Usage High Due to Heat

GERADO:
The stock market was up Tuesday morning after reports from Goldman Sachs showed profits rising faster than expected by investors who thought oil prices could go lower due to an oversupply problem. But stocks fell again Wednesday afternoon following disappointing quarterly results from General Electric.

--- Exemplo 2 ---
TRANSCRIÇÃO (primeiros 300 chars):
VAUSE: Welcome back, everybody. A short time ago, we got word that tennis executive, Raymond Moore, will step down as CEO of the BNP Paribas Open in California. That decision came after he made statements that stunned many people i

In [ ]:
# =======================================================================
# CÉLULA 10B — Backup provisório da inferência
# =======================================================================
import shutil
from google.colab import files

shutil.make_archive("./backup_inferencia_ft_mediasum_v2", "zip", OUTPUT_DIR, "resultados_finetuning.csv")
print(f"Tamanho: {os.path.getsize('./backup_inferencia_ft_mediasum_v2.zip') / (1024**2):.1f} MB")
files.download("./backup_inferencia_ft_mediasum_v2.zip")

Tamanho: 0.6 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =======================================================================
# CÉLULA 11 — Métricas e seleção das 40 amostras
# =======================================================================
if 'df_ft' not in dir():
    print("Carregando inferência do disco...")
    df_ft = pd.read_csv(f"{OUTPUT_DIR}/resultados_finetuning.csv")
    refs = df_ft["resumo_referencia"].fillna("").astype(str).tolist()
    gerados_ft = df_ft["resumo_gerado"].fillna("").astype(str).tolist()
    print(f"-> {len(df_ft)} resumos carregados!")

df_ft["resumo_gerado"] = df_ft["resumo_gerado"].fillna("").astype(str)
refs = df_ft["resumo_referencia"].fillna("").astype(str).tolist()
gerados_ft = df_ft["resumo_gerado"].tolist()

metricas_ft = calcular_metricas(refs, gerados_ft, prefixo="Fine-Tuning MediaSum v2")

pd.DataFrame([metricas_ft], index=["Fine-Tuning MediaSum v2"]).to_csv(
    f"{OUTPUT_DIR}/metricas_finetuning.csv"
)
print(pd.DataFrame([metricas_ft], index=["Fine-Tuning MediaSum v2"]).to_string())

def calcular_rouge_l_individual(row):
    result = rouge.compute(
        predictions=[row["resumo_gerado"]],
        references=[row["resumo_referencia"]]
    )
    return result["rougeL"]

print("\nCalculando ROUGE-L individual...")
df_ft["rouge_l_individual"] = df_ft.apply(calcular_rouge_l_individual, axis=1)

# Recupera índices do baseline
indices_df = pd.read_csv("./avaliacao_mediasum_v2/indices_baseline.csv")
indices_baseline = indices_df["indices"].tolist()
print(f"Índices do baseline recuperados: {len(indices_baseline)} amostras")

# Seleciona as mesmas 40 amostras do baseline
amostras_ft = df_ft.iloc[indices_baseline].copy().reset_index(drop=True)

for col in ["human_fluency", "human_coherence", "human_consistency", "human_relevance",
            "geval_fluency", "geval_coherence", "geval_consistency", "geval_relevance"]:
    amostras_ft[col] = ""

amostras_ft.to_csv(f"{OUTPUT_DIR}/amostras_finetuning_avaliacao_humana.csv", index=False)
print(f"40 amostras do FT salvas com sucesso!")


Calculando métricas Fine-Tuning MediaSum v2...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
BERTScore: 100%|██████████| 300/300 [00:10<00:00, 28.67it/s]


                         ROUGE-1  ROUGE-2  ROUGE-L   BLEU  METEOR  BERTScore-P  BERTScore-R  BERTScore-F1
Fine-Tuning MediaSum v2    0.108    0.026   0.0914  0.009  0.1206       0.9592       0.9685        0.9638

Calculando ROUGE-L individual...
Índices do baseline recuperados: 40 amostras
40 amostras do FT salvas com sucesso!


In [ ]:
# =======================================================================
# CÉLULA 12 — Gráficos comparativos
# =======================================================================
metricas_baseline = pd.read_csv(
    "./avaliacao_mediasum_v2/metricas_baseline.csv", index_col=0
).iloc[0].to_dict()

metricas_plot = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "METEOR", "BERTScore-F1"]
x = np.arange(len(metricas_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width/2, [metricas_baseline[m] for m in metricas_plot],
       width, label="Baseline", color="tab:blue")
ax.bar(x + width/2, [metricas_ft[m] for m in metricas_plot],
       width, label="Fine-Tuning MediaSum v2", color="tab:orange")

ax.set_xlabel("Métrica")
ax.set_ylabel("Score")
ax.set_title("BitNet b1.58 — Baseline vs Fine-Tuning (MediaSum v2)")
ax.set_xticks(x)
ax.set_xticklabels(metricas_plot)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/comparacao_metricas.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/comparacao_metricas.pdf", bbox_inches="tight")
plt.close()
print("Gráfico comparativo salvo!")

Gráfico comparativo salvo!


In [ ]:
# =======================================================================
# CÉLULA 13 — Verificação dos arquivos salvos
# =======================================================================
print("=== Arquivos na pasta ===")
for f in os.listdir(OUTPUT_DIR):
    tamanho = os.path.getsize(f"{OUTPUT_DIR}/{f}") / 1024
    print(f"{f} — {tamanho:.1f} KB")

print("\n=== metricas_finetuning.csv ===")
metricas = pd.read_csv(f"{OUTPUT_DIR}/metricas_finetuning.csv", index_col=0)
print(metricas.to_string())

print("\n=== amostras_finetuning_avaliacao_humana.csv ===")
amostras = pd.read_csv(f"{OUTPUT_DIR}/amostras_finetuning_avaliacao_humana.csv")
print(f"Linhas: {len(amostras)}")
print(f"Colunas: {amostras.columns.tolist()}")

=== Arquivos na pasta ===
comparacao_metricas.png — 101.1 KB
resultados_finetuning.csv — 1564.0 KB
comparacao_metricas.pdf — 16.2 KB
amostras_finetuning_avaliacao_humana.csv — 202.7 KB
metricas_finetuning.csv — 0.1 KB

=== metricas_finetuning.csv ===
                         ROUGE-1  ROUGE-2  ROUGE-L   BLEU  METEOR  BERTScore-P  BERTScore-R  BERTScore-F1
Fine-Tuning MediaSum v2    0.108    0.026   0.0914  0.009  0.1206       0.9592       0.9685        0.9638

=== amostras_finetuning_avaliacao_humana.csv ===
Linhas: 40
Colunas: ['dialogo', 'resumo_referencia', 'resumo_gerado', 'rouge_l_individual', 'human_fluency', 'human_coherence', 'human_consistency', 'human_relevance', 'geval_fluency', 'geval_coherence', 'geval_consistency', 'geval_relevance']


In [ ]:
# =======================================================================
# CÉLULA 14 — Backup completo do FT MediaSum v2
# =======================================================================
import shutil
from google.colab import files

os.makedirs("./backup_finetuning_mediasum_v2", exist_ok=True)
shutil.copytree(OUTPUT_DIR, "./backup_finetuning_mediasum_v2/avaliacao_mediasum_v2_ft", dirs_exist_ok=True)
shutil.make_archive("./backup_finetuning_mediasum_v2", "zip", "./backup_finetuning_mediasum_v2")
print(f"Backup gerado: backup_finetuning_mediasum_v2.zip ({os.path.getsize('./backup_finetuning_mediasum_v2.zip') / (1024**2):.1f} MB)")
files.download("./backup_finetuning_mediasum_v2.zip")

Backup gerado: backup_finetuning_mediasum_v2.zip (0.7 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>